# Traefik — Docker Ingress & Let's Encrypt

Traefik discovers services via Docker labels and issues TLS certs automatically via ACME/Let's Encrypt.

## Generate traefik.yml static config

In [ ]:
import yaml

def traefik_static(http_port=80, https_port=443, dashboard_port=8080,
                   acme_email="admin@example.com", log_level="INFO"):
    return {
        "api": {"dashboard": True, "insecure": True},
        "entryPoints": {
            "web":      {"address": f":{http_port}",
                         "http": {"redirections": {"entryPoint": {"to": "websecure", "scheme": "https"}}}},
            "websecure": {"address": f":{https_port}"},
        },
        "providers": {
            "docker": {"endpoint": "unix:///var/run/docker.sock", "exposedByDefault": False},
            "file":   {"filename": "/etc/traefik/dynamic.yml", "watch": True},
        },
        "certificatesResolvers": {
            "letsencrypt": {"acme": {
                "email": acme_email,
                "storage": "/letsencrypt/acme.json",
                "httpChallenge": {"entryPoint": "web"},
            }}
        },
        "log": {"level": log_level},
        "accessLog": {},
    }

print(yaml.dump(traefik_static(), default_flow_style=False, sort_keys=False))

## Generate Docker Compose labels for a service

In [ ]:
def traefik_labels(service_name, hostname, container_port, tls=True, resolver="letsencrypt"):
    base = f"traefik.http"
    labels = {
        "traefik.enable": "true",
        f"{base}.routers.{service_name}.rule": f"Host(`{hostname}`)",
        f"{base}.routers.{service_name}.entrypoints": "websecure" if tls else "web",
        f"{base}.services.{service_name}.loadbalancer.server.port": str(container_port),
    }
    if tls:
        labels[f"{base}.routers.{service_name}.tls.certresolver"] = resolver
    return labels

labels = traefik_labels("myapp", "app.example.com", 3000)
print("labels:")
for k, v in labels.items():
    print(f"  - \"{k}={v}\"")

## Middleware builder

In [ ]:
def rate_limit_middleware(name, avg=100, burst=50):
    return {
        "http": {"middlewares": {
            name: {"rateLimit": {"average": avg, "burst": burst}}
        }}
    }

def basic_auth_middleware(name, users):
    return {
        "http": {"middlewares": {
            name: {"basicAuth": {"users": users}}
        }}
    }

import yaml
print("# Rate limiting middleware")
print(yaml.dump(rate_limit_middleware("api-rate-limit", avg=50, burst=20),
                default_flow_style=False))

print("# Basic auth middleware")
print(yaml.dump(basic_auth_middleware("admin-auth", ["admin:$apr1$...$hashedpassword"]),
                default_flow_style=False))